In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os, sys 

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

# The master run id ties every log line from this task back to the pipeline run
# that produced it. Without it a mart event - the orphaned-fact count in
# particular - cannot be attributed to a run, which is most of its value.
master_run_id = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="master_run_id")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -----------------------------
# Add shared utilities path
# -----------------------------
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")

from superstore_placeholder_monitor import log_placeholder_exposure
# Imported unconditionally, NOT inside the alert branch. An import that only
# runs when something is already wrong is an import that has never run: a
# missing module would surface during an incident rather than on this line.
from superstore_alerting import get_webhook_url, route_alert, should_alert
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog_dev / _qa / _prod
gold_schema = get_gold_schema()    # e.g., superstore_gold
mart_schema = get_mart_schema() 

from superstore_logger import get_superstore_logger, log_event
from superstore_backfill_utils import is_dry_run
# -----------------------------
# log_event Setup 
# -----------------------------
# Initialize log_event to capture events 
logger_mart_customers_360 = get_superstore_logger("superstore_mart_customers_360")


# -----------------------------
# Dry run
# -----------------------------
# The serving layer rebuilds from Gold and has no run-mode logic of its own —
# a replay or backfill simply produces different Gold for it to read. But it
# does write, so it must not write during a dry run. is_dry_run reads only the
# dry_run parameter and never raises, so a serving task cannot fail on the
# full_refresh gate that is enforced upstream where it belongs.
if is_dry_run(dbutils):
    log_event(
        logger_mart_customers_360,
        "INFO",
        "Dry run: skipping mart_customer_360 write",
        dry_run=True,
        master_run_id=master_run_id,
        layer="Mart"
    )
    dbutils.notebook.exit("DRY_RUN_COMPLETED layer=mart_customer_360 (no data written)")

# -------------------------------
# Function to check if the table exists
# -------------------------------
def check_table_exists(catalog, schema, table_name):
    try:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        return spark.catalog.tableExists(full_table_name)
    except Exception as e:
        log_event(
            logger_mart_customers_360,
            "ERROR",
            f"Error checking table existence: {e}",
            layer="Feature Engineering"
        )
        return False

# -------------------------------
# Required table to create or update features Table (Environment-aware)
# -------------------------------
# Check if the required tables exist before creating the feature table
required_tables = [
    f"{catalog}.{gold_schema}.dim_customers",
    f"{catalog}.{gold_schema}.facts_orders",
    f"{catalog}.{gold_schema}.facts_sales"
]

missing_tables = []

# Check each table
for table in required_tables:
    if not check_table_exists(catalog, gold_schema, table.split('.')[-1]):
        missing_tables.append(table)

if missing_tables:
    log_event(logger_mart_customers_360, "ERROR", f"Missing tables: {', '.join(missing_tables)}. Cannot create mart customer 360 table.", master_run_id=master_run_id, layer="Mart")
else:
    # -------------------------------
    # Referential completeness
    # -------------------------------
    # This mart is built from Gold facts and dimensions joined on the CURRENT
    # dimension version. A fact whose dimension row is absent contributes nothing
    # to the mart, and nothing reports it: the row is not quarantined, not
    # audited, and not counted anywhere. It is simply not in the output.
    #
    # The reconciliation invariant does not cover this. It proves
    # bronze == silver + quarantine + audit, which is a SILVER guarantee that
    # every row is accounted for - not that it is usable downstream. A customer
    # quarantined for one bad attribute never reaches the dimension, so their
    # orders are dropped here while every upstream check still passes.
    #
    # Counting it does not fix it. It makes a silent loss a monitored one, which
    # is the same reason quarantine records the rule a row violated.

    _orphans_customers = spark.sql(f"""
        SELECT COUNT(*) AS n
        FROM {catalog}.{gold_schema}.facts_orders f
        LEFT ANTI JOIN {catalog}.{gold_schema}.dim_customers d
          ON f.customer_id = d.customer_id AND d.is_current = true
    """).first()["n"]

    log_event(
        logger_mart_customers_360,
        "WARN" if _orphans_customers else "INFO",
        f"facts_orders rows with no current dim_customers version: {_orphans_customers}",
        orphaned_facts=_orphans_customers,
        fact_table="facts_orders",
        dimension_table="dim_customers",
        master_run_id=master_run_id,
        layer="Mart"
    )

    # Orphaned facts are structurally 0 since severity tiers, so a non-zero
    # count means a NEW cause -- one nobody has thought of. That is exactly the
    # case worth waking someone for, and exactly the case that previously only
    # reached a log line nobody read.
    if _orphans_customers:
        if should_alert("orphaned_facts", _orphans_customers):
            route_alert(
                logger_mart_customers_360,
                get_webhook_url(dbutils),
                "ERROR",
                "Orphaned facts detected in facts_orders",
                "Facts reference a dim_customers member that does not exist, so their "
                "revenue is absent from every mart.",
                fields={"orphaned_facts": _orphans_customers, "fact_table": "facts_orders",
                        "dimension_table": "dim_customers"},
                layer="Mart",
                master_run_id=master_run_id,
            )

    # -------------------------------
    # Placeholder exposure
    # -------------------------------
    # Successor to the orphan count above. Severity tiers made that count
    # structurally 0 -- a quarantined dimension no longer strands its facts --
    # so it now reads 0 whether the source is clean or badly incomplete.
    #
    # The failure moved rather than disappeared: revenue that used to be
    # ABSENT is now ATTRIBUTED to a bucket carrying no information. Watch the
    # trend, not the absolute number -- some entities genuinely have no value
    # anywhere, but a figure climbing run over run means a source started
    # arriving incomplete.
    log_placeholder_exposure(
        spark,
        logger_mart_customers_360,
        catalog=catalog,
        schema=gold_schema,
        dimension_table="dim_customers",
        attribute_columns=['customer_name', 'segment', 'country', 'state', 'city', 'postal_code', 'region'],
        master_run_id=master_run_id,
    )

    # -------------------------------
    # Create mart Table (Environment-aware)
    # -------------------------------
    spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.{mart_schema}.mart_customer_360 AS

    WITH customer_orders AS (
        SELECT
            c.customer_id,
            c.customer_name,
            c.segment,
            c.region,

            SUM(s.sales) AS total_sales,
            SUM(s.profit) AS total_profit,
            COUNT(DISTINCT o.order_id) AS total_orders,
            AVG(s.discount) AS discount_sensitivity,

            MAX(o.order_date) AS last_order_date,
            MIN(o.order_date) AS first_order_date

        FROM {catalog}.{gold_schema}.dim_customers c

        -- FIRST join orders (correct grain)
        LEFT JOIN {catalog}.{gold_schema}.facts_orders o
            ON c.customer_id = o.customer_id

        -- THEN join sales using order_id
        LEFT JOIN {catalog}.{gold_schema}.facts_sales s
            ON o.order_id = s.order_id

        WHERE c.is_current = TRUE

        GROUP BY 
            c.customer_id, 
            c.customer_name, 
            c.segment, 
            c.region
    )

    SELECT
        customer_id,
        customer_name,
        segment,
        region,
        total_sales,
        total_profit,
        total_orders,

        CASE 
            WHEN total_orders = 0 THEN NULL 
            ELSE ROUND(total_sales / total_orders, 2) 
        END AS avg_order_value,

        last_order_date,
        DATEDIFF(current_date(), last_order_date) AS recency_days,
        DATEDIFF(last_order_date, first_order_date) AS customer_lifetime_days,
        discount_sensitivity,

        CASE 
            WHEN total_profit > 0 THEN 1 
            ELSE 0 
        END AS profitability_flag

    FROM customer_orders
    """)
log_event(logger_mart_customers_360, "INFO", "Mart customer 360 table created successfully.", master_run_id=master_run_id, layer="Mart")

